# Учительская проверка submission: итоговое соревнование ансамблей

Этот блокнот не нужно раздавать участникам вместе с ответами. Он читает файлы команд из папки `submissions` и сравнивает их со скрытым `data/private_target.csv`.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score

DATA_DIR = Path("data")
SUBMISSIONS_DIR = Path("submissions")

private = pd.read_csv(DATA_DIR / "private_target.csv")
files = sorted(SUBMISSIONS_DIR.glob("*.csv"))

rows = []
for path in files:
    sub = pd.read_csv(path)
    merged = private.merge(sub, on="id", how="inner")
    if len(merged) != len(private):
        print(f"Пропускаю {path.name}: найдено {len(merged)} id из {len(private)}")
        continue
    if "churn_probability" not in merged.columns:
        print(f"Пропускаю {path.name}: нужен столбец churn_probability")
        continue
    y_true = merged["churn"]
    proba = merged["churn_probability"].clip(0, 1)
    pred = (proba >= 0.5).astype(int)
    rows.append(
        {
            "team_file": path.name,
            "ROC_AUC": roc_auc_score(y_true, proba),
            "average_precision": average_precision_score(y_true, proba),
            "F1_at_0_5": f1_score(y_true, pred, zero_division=0),
            "precision_at_0_5": precision_score(y_true, pred, zero_division=0),
            "recall_at_0_5": recall_score(y_true, pred, zero_division=0),
        }
    )

if rows:
    leaderboard = pd.DataFrame(rows).sort_values(["ROC_AUC", "average_precision"], ascending=[False, False])
    display(leaderboard)
else:
    print("Положите CSV-файлы команд в папку submissions и запустите ячейку снова.")
